# LAB | Day 2 | Data Manipulation + JSON Handling
## John Adams

### Step 1: Setting Up the Project

In [1]:
"""
Titanic Data Analysis and JSON Export
Description: Analyze Titanic passenger data, engineer features, and export to JSON
"""
 
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path
 
# Set up paths
DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_data.json"
 
# Create data directory if it doesn't exist
DATA_DIR.mkdir(exist_ok=True)
 
print("Project setup complete!")
print(f"Data directory: {DATA_DIR}")
print(f"CSV file location: {CSV_FILE}")

Project setup complete!
Data directory: data
CSV file location: data/titanic.csv


### Step 2: Importing and Exploring the Data

In [2]:
# Get the data and put it into a dataframe (table)
df = pd.read_csv(CSV_FILE)
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


### Step 3: Calculating Descriptive Statistics

In [56]:
# Select numeric columns 
numeric_columns = df.select_dtypes(include=['number']).columns

# Calculate statistics (.mean, median, std)
print("Mean:")
print(df[numeric_columns].mean())
print()

print("Median:")
print(df[numeric_columns].median())
print()

print("Standard Deviation:")
print(df[numeric_columns].std())
print()

# Table view
print("="*100)
print("A more fancy way to view the data set")
print("="*100)
df.describe()

Mean:
PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
Parch            0.381594
Fare            32.204208
dtype: float64

Median:
PassengerId    446.0000
Survived         0.0000
Pclass           3.0000
Age             28.0000
SibSp            0.0000
Parch            0.0000
Fare            14.4542
dtype: float64

Standard Deviation:
PassengerId    257.353842
Survived         0.486592
Pclass           0.836071
Age             14.526497
SibSp            1.102743
Parch            0.806057
Fare            49.693429
dtype: float64

A more fancy way to view the data set


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


### Step 4: Identifying Missing Values

In [55]:
# Count missing values
print("="*50)
print("MISSING VALUES ANALYSIS")
print("="*50)
 
missing_data = {}
 
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percent = (missing_count / len(df)) * 100

    print(f"Column: {col}, Missing Values: {missing_count}, Percentage: {missing_percent:.2f}%")

MISSING VALUES ANALYSIS
Column: PassengerId, Missing Values: 0, Percentage: 0.00%
Column: Survived, Missing Values: 0, Percentage: 0.00%
Column: Pclass, Missing Values: 0, Percentage: 0.00%
Column: Name, Missing Values: 0, Percentage: 0.00%
Column: Sex, Missing Values: 0, Percentage: 0.00%
Column: Age, Missing Values: 177, Percentage: 19.87%
Column: SibSp, Missing Values: 0, Percentage: 0.00%
Column: Parch, Missing Values: 0, Percentage: 0.00%
Column: Ticket, Missing Values: 0, Percentage: 0.00%
Column: Fare, Missing Values: 0, Percentage: 0.00%
Column: Cabin, Missing Values: 687, Percentage: 77.10%
Column: Embarked, Missing Values: 2, Percentage: 0.22%


### Step 5: Feature Engineering

In [6]:
# Create a copy of the dataframe for feature engineering
df_features = df.copy()
 
# Feature 1: Family Size
df_features['FamilySize'] = df_features['SibSp'] + df_features['Parch'] + 1
print(df_features[['SibSp', 'Parch', 'FamilySize']].head(10))
 
# Feature 2: Is Alone
df_features['IsAlone'] = df_features['FamilySize'] == 1
print(df_features[['FamilySize', 'IsAlone']].head(10))
 
# Feature 3: Age Groups
def categorize_age(age):
    """Categorize age into groups"""
    if pd.isna(age):
        return 'Unknown'
    elif age < 18:
        return 'Minor'
    elif age < 30:
        return 'Young Adult'
    elif age < 50:
        return 'Adult'
    else:
        return 'Senior'
 
df_features['AgeGroup'] = df_features['Age'].apply(categorize_age)
print(df_features[['Age', 'AgeGroup']].head(10))
 
# Analyze feature differences between survivors and non-survivors
print("\n" + "="*50)
print("FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED")
print("="*50)
 
# Here is an example to get you started:
print("\nFamily Size by Survival:")
family_survival = df_features.groupby('Survived')['FamilySize'].agg(['mean', 'median', 'std'])
print(family_survival)
 
# Statistical test: Do these features help differentiate?
print("\n" + "="*50)
print("FEATURE DIFFERENTIATION ANALYSIS")
print("="*50)
 
survived = df_features[df_features['Survived'] == 1]
not_survived = df_features[df_features['Survived'] == 0]
 
print("\nFamily Size:")
print(f"  Survived mean: {survived['FamilySize'].mean():.2f}")
print(f"  Not Survived mean: {not_survived['FamilySize'].mean():.2f}")
print(f"  Difference: {abs(survived['FamilySize'].mean() - not_survived['FamilySize'].mean()):.2f}")

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2
   FamilySize  IsAlone
0           2    False
1           2    False
2           1     True
3           2    False
4           1     True
5           1     True
6           1     True
7           5    False
8           3    False
9           2    False
    Age     AgeGroup
0  22.0  Young Adult
1  38.0        Adult
2  26.0  Young Adult
3  35.0        Adult
4  35.0        Adult
5   NaN      Unknown
6  54.0       Senior
7   2.0        Minor
8  27.0  Young Adult
9  14.0        Minor

FEATURE ANALYSIS: SURVIVED vs NOT SURVIVED

Family Size by Survival:
              mean  median       std
Survived                            
0         1.883424     1.0  1.830669
1         1.938596     2.0 

### Step 6: Creating a Data Export Class

In [59]:
# Step 3: Create Classes for JSON Export
 
class Passenger:
    """
    Represents a passenger with all their information.
    """
    def __init__(self, passenger_id, name, age, sex, survived, pclass, 
                 fare, embarked=None, family_size=None, is_alone=None, title=None):
        # TODO: Initialize all passenger attributes
        # Tip: Use pd.notna() to check if a value is not null/NaN
        # Tip: Convert to appropriate types (int, float, str)
        # Example for passenger_id:
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name = str(name) if pd.notna(name) else None
        self.age = float(age) if pd.notna(age) else None
        self.sex = str(sex) if pd.notna(sex) else None
        self.survived = int(survived) if pd.notna(survived) else None
        self.pclass = int(pclass) if pd.notna(pclass) else None
        self.fare = float(fare) if pd.notna(fare) else None
        self.embarked = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone = bool(is_alone) if pd.notna(is_alone) else None
        self.title = str(title) if pd.notna(title) else None

        # TODO: Complete initialization for remaining attributes
        # Remember to handle None/NaN values appropriately
        pass
    
    def to_dict(self):
        """Convert passenger to dictionary for JSON serialization."""
        # TODO: Return a dictionary with all passenger attributes
        # Tip: Dictionary keys should match the attribute names
        return {
            # TODO: Add all other attributes here
            'passenger_id': self.passenger_id,
            'name': self.name,
            'age': self.age,
            'sex': self.sex,
            'survived': self.survived,
            'pclass': self.pclass,
            'fare': self.fare,
            'embarked': self.embarked,
            'family_size': self.family_size,
            'is_alone': self.is_alone,
            'title': self.title
        }
 
class TitanicDataset:
    """
    Represents the entire Titanic dataset with methods for JSON export.
    """
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.passengers = []  # Will store Passenger objects
        self._create_passengers()
    
    def _create_passengers(self):
        """Create Passenger objects from dataframe."""
        # TODO: Iterate through dataframe rows and create Passenger objects
        # Tip: Use self.dataframe.iterrows() to loop through rows
        # Tip: Use row.get('ColumnName', default_value) to safely get values
        for idx, row in self.dataframe.iterrows():
            # TODO: Create a Passenger object with data from the row
            # Example of getting PassengerId:
            # passenger_id = row.get('PassengerId', idx)
            
            passenger_id = row.get('PassengerId', idx)
            name = row.get('Name', None)
            age = row.get('Age', None)
            sex = row.get('Sex', None)
            survived = row.get('Survived', None)
            pclass = row.get('Pclass', None)
            fare = row.get('Fare', None)
            embarked = row.get('Embarked', None)
            family_size = row.get('FamilySize', None)
            is_alone = row.get('IsAlone', None)
            title = row.get('Title', None)

            # TODO: Create the Passenger object and append to self.passengers
            pass
            new_passenger = Passenger(passenger_id, name, age, sex, survived, pclass, fare, embarked, family_size, is_alone, title)
            self.passengers.append(new_passenger)
    
    def to_json(self, filename='titanic_data.json'):
        """Export dataset to JSON file."""
        # TODO: Create a dictionary with metadata and passenger data
        data = {
            'metadata': {
                'dataset_name': 'Titanic Passenger Dataset',
                'export_date': datetime.now().isoformat(),
                # TODO: Add more metadata fields
                # Tip: Calculate total_passengers from self.passengers
                # Tip: Calculate survival_rate using self.dataframe['Survived'].mean()
                'total_passengers': len(self.passengers),
                'survival_rate': self.dataframe['Survived'].mean() if not self.dataframe.empty else 0
            },
            'passengers': [p.to_dict() for p in self.passengers]  # TODO: Convert all passenger objects to dictionaries
            # Tip: Use list comprehension with p.to_dict() for each passenger
        }
        
        # TODO: Write the data to a JSON file
        # Tip: Use json.dump() with indent=2 for readable formatting
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2)
        
        print(f"Data exported to {filename}")
        return data
    
    def get_summary_stats(self):
        """Get summary statistics."""
        # TODO: Calculate and return summary statistics
        # Tip: Use list comprehensions to filter and calculate
        # Example for counting survived passengers:
        # survived_count = sum(1 for p in self.passengers if p.survived == 1)
        survived_count = sum(1 for p in self.passengers if p.survived == 1)
        did_not_survive_count = sum(1 for p in self.passengers if p.survived == 0)
        total_passengers = len(self.passengers)
        
        return {
            'total_passengers': total_passengers,  # TODO: Calculate total passengers
            'survived': survived_count,  # TODO: Count passengers who survived
            'did_not_survive': did_not_survive_count,  # TODO: Count passengers who didn't survive
            # TODO: Add more statistics (average_age, average_fare)
            # Tip: Remember to handle None values when calculating averages
            'average_age': np.mean([p.age for p in self.passengers if p.age is not None]) if total_passengers > 0 else None,
            'average_fare': np.mean([p.fare for p in self.passengers if p.fare is not None]) if total_passengers > 0 else None
        }
 
# TODO: Create dataset object and export
# Check if df_engineered exists and is not empty
if 'df_features' in locals() and not df_features.empty:
    # TODO: Create a TitanicDataset object
    # dataset = TitanicDataset(...)
    dataset = TitanicDataset(df_features)
    
    # TODO: Print basic information about the dataset
    print("Dataset Information:")
    print(f"Total passengers: {len(dataset.passengers)}")
    print(f"Survival rate: {dataset.dataframe['Survived'].mean()}")
    print()
    
    # TODO: Get and display summary statistics
    # Tip: Call get_summary_stats() and iterate through the results
    get_summary = dataset.get_summary_stats()
    for key, value in get_summary.items():
        print(f"{key}: {value}")
    
    # TODO: Export to JSON (optional - uncomment when ready)
    # dataset.to_json('titanic_data.json')
    dataset.to_json(JSON_FILE)

Dataset Information:
Total passengers: 891
Survival rate: 0.3838383838383838

total_passengers: 891
survived: 342
did_not_survive: 549
average_age: 29.69911764705882
average_fare: 32.204207968574636
Data exported to data/titanic_data.json


### Step 7: Testing and Validation

In [58]:
# Additional validation: Load and inspect JSON
with open(JSON_FILE, 'r', encoding='utf-8') as f:
    json_data = json.load(f)
 
# Print summary of JSON data and verify content
print("Top-level keys:", list(json_data.keys()))
print("Metadata:", json_data['metadata'])
print("Number of passenger records:", len(json_data['passengers']))


Top-level keys: ['metadata', 'passengers']
Metadata: {'dataset_name': 'Titanic Passenger Dataset', 'export_date': '2026-07-09T16:35:08.778947', 'total_passengers': 891, 'survival_rate': 0.3838383838383838}
Number of passenger records: 891
